# Document-level CID relation extraction (Zheng et al., 2018)

Goal: for every (chemical, disease) pair in a PubMed abstract, predict whether the chemical
causes the disease (CID = chemical-induced disease).

Paper: *An effective neural model extracting document level chemical-induced disease relations
from biomedical literature* (JBI 83, 2018). DOI: https://doi.org/10.1016/j.jbi.2018.05.001

This notebook follows the paper pipeline (preprocessing, BiLSTM+CNN model, post-processing rules)
using **local CDR data** under `Dataset/` and **Stanford CoreNLP** for sentence split / tokenize / POS
(paper §2.2.2). If Java + CoreNLP are not installed, it falls back to **Stanza** (Stanford NLP Group)
so the notebook still runs.

## Part 1. Setup

### CoreNLP (paper-faithful)

This machine is set up with:

1. **JDK 11** at `C:\Program Files\Java\jdk-11.0.33.0.3` (auto-detected if not on `PATH`).
2. **Stanford CoreNLP 4.6.0** in `stanford-corenlp-4.6.0/` (auto-detected as `CORENLP_HOME`).
3. **English-extra models** in `corenlp-english-extra/` (copied into the CoreNLP folder on first run).

You can still override with env vars `JAVA_HOME` / `CORENLP_HOME`.

GPU (optional): Colab *Runtime → GPU*, or a local CUDA TensorFlow build.

In [28]:
# Run once per environment
%pip install -q tensorflow gensim scikit-learn stanfordcorenlp stanza

/home/oxored/Documents/DocCIDR/dl/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [29]:
import os, re, random, zipfile, urllib.request, json, subprocess, shutil, gc
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from gensim.models import Word2Vec

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# ---- GPU: growth so TF does not grab the whole 6 GB at once ----
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print("memory_growth:", e)

# ---- Settings from the paper (section 3.3) + recall-oriented tweaks ----
K_WINDOW      = 4
MAX_SENTS     = 30
MAX_WORDS     = 120
WORD_DIM      = 100
POS_DIM       = 10
CNN_WINDOW    = 5
CNN_FILTERS   = 300
DROPOUT       = 0.5
LEARNING_RATE = 0.001
BATCH_SIZE    = 4          # was 8; safer on 6 GB laptop GPU
THRESHOLD     = 0.3        # best from recall retrain (dev F1); still swept below
MAX_EPOCHS    = 40         # was 20
PATIENCE      = 8          # was 3
# Up-weight CID positives (~19% of train) to push recall
POS_WEIGHT    = None       # None → balanced  total/(2*count); or set e.g. 4.0
THRESH_GRID   = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25]

REPO_ROOT = Path(".").resolve()
CDR_DIR = REPO_ROOT / "Dataset" / "CDR_Data" / "CDR_Data" / "CDR.Corpus.v010516"
EXTRA_MODELS = REPO_ROOT / "corenlp-english-extra" / "stanford-corenlp-models-english-extra.jar"
CORENLP_DIR = REPO_ROOT / "stanford-corenlp-4.6.0"

def ensure_java_on_path():
    """Put JDK 11 on PATH for this process if `java` is missing."""
    if shutil.which("java"):
        return shutil.which("java")
    candidates = []
    if os.environ.get("JAVA_HOME"):
        candidates.append(Path(os.environ["JAVA_HOME"]) / "bin" / "java.exe")
    candidates += [
        Path(r"C:\Program Files\Java\jdk-11.0.33.0.3\bin\java.exe"),
        Path(r"C:\Program Files\Java\latest\bin\java.exe"),
    ]
    java_root = Path(r"C:\Program Files\Java")
    if java_root.is_dir():
        candidates += sorted(java_root.glob("jdk-11*\\bin\\java.exe"), reverse=True)
    for java_exe in candidates:
        if java_exe.is_file():
            bin_dir = str(java_exe.parent)
            os.environ["JAVA_HOME"] = str(java_exe.parent.parent)
            os.environ["PATH"] = bin_dir + os.pathsep + os.environ.get("PATH", "")
            return str(java_exe)
    return None

JAVA_EXE = ensure_java_on_path()
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
print("CDR_DIR exists:", CDR_DIR.is_dir())
print("CoreNLP dir:", CORENLP_DIR.is_dir(), CORENLP_DIR)
print("english-extra jar:", EXTRA_MODELS.is_file())
print("java:", JAVA_EXE or "NOT FOUND")
if JAVA_EXE:
    print(subprocess.check_output([JAVA_EXE, "-version"], stderr=subprocess.STDOUT, text=True).splitlines()[0])


TF 2.21.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
CDR_DIR exists: True
CoreNLP dir: True /home/oxored/Documents/DocCIDR/stanford-corenlp-4.6.0
english-extra jar: True
java: /usr/bin/java
openjdk version "11.0.33.1" 2026-11-17


## Part 2. Data — BioCreative V CDR (local)

1,500 PubMed abstracts (500 / 500 / 500). PubTator format:

- `PMID|t|title` and `PMID|a|abstract`
- entity: `PMID\tstart\tend\tmention\ttype\tMeSH`
- relation: `PMID\tCID\tchem-MeSH\tdisease-MeSH`

Paths: `Dataset/CDR_Data/CDR_Data/CDR.Corpus.v010516/`.

In [30]:
TRAIN_FILE = CDR_DIR / "CDR_TrainingSet.PubTator.txt"
DEV_FILE   = CDR_DIR / "CDR_DevelopmentSet.PubTator.txt"
TEST_FILE  = CDR_DIR / "CDR_TestSet.PubTator.txt"

for p in (TRAIN_FILE, DEV_FILE, TEST_FILE):
    assert p.is_file(), f"Missing {p}. Place the CDR corpus under Dataset/."
    print(p)

/home/oxored/Documents/DocCIDR/Dataset/CDR_Data/CDR_Data/CDR.Corpus.v010516/CDR_TrainingSet.PubTator.txt
/home/oxored/Documents/DocCIDR/Dataset/CDR_Data/CDR_Data/CDR.Corpus.v010516/CDR_DevelopmentSet.PubTator.txt
/home/oxored/Documents/DocCIDR/Dataset/CDR_Data/CDR_Data/CDR.Corpus.v010516/CDR_TestSet.PubTator.txt


In [31]:
def read_pubtator(path):
    """Return a list of documents. Each document is a dict."""
    docs, doc = [], None
    path = Path(path)
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.rstrip("\n")
        if line == "":
            if doc:
                docs.append(doc)
            doc = None
        elif re.match(r"^\d+\|t\|", line):
            pmid, _, title = line.split("|", 2)
            doc = {"pmid": pmid, "title": title, "abstract": "",
                   "entities": [], "cids": set()}
        elif re.match(r"^\d+\|a\|", line):
            doc["abstract"] = line.split("|", 2)[2]
        else:
            parts = line.split("\t")
            if len(parts) < 2:
                continue
            if parts[1] == "CID":
                doc["cids"].add((parts[2], parts[3]))
            else:
                mesh = parts[5] if len(parts) > 5 else ""
                ids = [i for i in mesh.split("|") if i and i != "-1"]
                if ids:
                    doc["entities"].append({
                        "start": int(parts[1]),
                        "end": int(parts[2]),
                        "text": parts[3],
                        "type": parts[4],
                        "ids": ids,
                    })
    if doc:
        docs.append(doc)
    return docs

train_docs = read_pubtator(TRAIN_FILE)
dev_docs   = read_pubtator(DEV_FILE)
test_docs  = read_pubtator(TEST_FILE)

print(len(train_docs), len(dev_docs), len(test_docs))
print(sum(len(d["cids"]) for d in train_docs + dev_docs + test_docs), "CIDs")

500 500 500
3116 CIDs


## Part 3. Preprocessing (paper §2.2.2 and §2.2.4)

1. Split title + abstract into sentences (title = sentence 0).
2. Map each entity mention to a MeSH token.
3. Replace numbers by `<num>`; POS-tag every token (CoreNLP / Stanza).
4. For each candidate pair: wrap chemical in `ch_start … ch_end`, disease in `ds_start … ds_end`;
   drop brackets that hold neither candidate.
5. On the new development set and the test set only, drop pairs whose minimum sentence span exceeds **K = 4** (§3.1). Training keeps every pair.


In [32]:
# ---------- NLP backend: CoreNLP (preferred) or Stanza ----------

class _StanzaBackend:
    """Stanford Stanza fallback when Java/CoreNLP is unavailable."""
    def __init__(self):
        import stanza
        stanza.download("en", processors="tokenize,pos", verbose=False)
        self.nlp = stanza.Pipeline(
            "en", processors="tokenize,pos", tokenize_no_ssplit=False, verbose=False
        )

    def annotate(self, text):
        doc = self.nlp(text)
        sentences = []
        for sent in doc.sentences:
            tokens = []
            for w in sent.words:
                start = int(w.start_char) if w.start_char is not None else -1
                end = int(w.end_char) if w.end_char is not None else -1
                tokens.append({"word": w.text, "pos": w.xpos or w.upos or "NN",
                               "start": start, "end": end})
            sentences.append(tokens)
        return sentences


class _CoreNLPBackend:
    """Stanford CoreNLP via stanfordcorenlp (Java server)."""
    def __init__(self, home: Path):
        from stanfordcorenlp import StanfordCoreNLP
        jars = list(Path(home).glob("stanford-corenlp-*.jar"))
        if not jars:
            raise FileNotFoundError(f"No stanford-corenlp-*.jar under {home}")
        # Prefer putting english-extra on the classpath by copying/linking into home if present
        if EXTRA_MODELS.is_file():
            dest = Path(home) / EXTRA_MODELS.name
            if not dest.exists():
                try:
                    shutil.copy2(EXTRA_MODELS, dest)
                except Exception:
                    pass
        self.nlp = StanfordCoreNLP(str(home), memory="4g", timeout=60000)
        self.props = {"annotators": "tokenize,ssplit,pos", "outputFormat": "json",
                      "tokenize.options": "ptb3Escaping=false"}

    def annotate(self, text):
        raw = self.nlp.annotate(text, properties=self.props)
        data = json.loads(raw)
        sentences = []
        for sent in data.get("sentences", []):
            tokens = []
            for t in sent.get("tokens", []):
                tokens.append({
                    "word": t.get("originalText") or t.get("word"),
                    "pos": t.get("pos", "NN"),
                    "start": int(t["characterOffsetBegin"]),
                    "end": int(t["characterOffsetEnd"]),
                })
            sentences.append(tokens)
        return sentences

    def close(self):
        try:
            self.nlp.close()
        except Exception:
            pass


def _java_ok():
    # ensure_java_on_path() already ran in the settings cell
    return shutil.which("java") is not None or JAVA_EXE is not None


def _find_corenlp_home():
    env = os.environ.get("CORENLP_HOME")
    if env and Path(env).is_dir():
        return Path(env)
    for cand in [
        CORENLP_DIR,  # DocCIDR/stanford-corenlp-4.6.0
        REPO_ROOT / "stanford-corenlp-4.6.0",
        REPO_ROOT / "stanford-corenlp",
        REPO_ROOT / "corenlp",
        Path.home() / "stanford-corenlp",
    ]:
        if cand.is_dir() and list(cand.glob("stanford-corenlp-*.jar")):
            return cand
    return None


def build_nlp_backend():
    ensure_java_on_path()
    home = _find_corenlp_home()
    if _java_ok() and home is not None:
        os.environ["CORENLP_HOME"] = str(home)
        print("Using Stanford CoreNLP at", home)
        return _CoreNLPBackend(home), "corenlp"
    print("Java/CoreNLP not ready — using Stanza (Stanford NLP) fallback.")
    print("Expected CoreNLP folder:", CORENLP_DIR)
    print("java on PATH:", shutil.which("java"), "| JAVA_EXE:", JAVA_EXE)
    return _StanzaBackend(), "stanza"


NLP, NLP_BACKEND = build_nlp_backend()
print("NLP_BACKEND =", NLP_BACKEND)
assert NLP_BACKEND == "corenlp", (
    "CoreNLP did not start. Check JDK on PATH / JAVA_HOME and folder stanford-corenlp-4.6.0/."
)

Using Stanford CoreNLP at /home/oxored/Documents/DocCIDR/stanford-corenlp-4.6.0
NLP_BACKEND = corenlp


In [33]:
def normalize_word(w):
    w = w.lower()
    if re.fullmatch(r"[\d.,]*\d[\d.,]*", w):
        return "<num>"
    return w


def _merge_entities_into_sentence(abs_tokens, entities):
    """Replace token spans that overlap gold entities with MeSH placeholders."""
    merged = []
    i = 0
    while i < len(abs_tokens):
        t = abs_tokens[i]
        hit = None
        if t["start"] >= 0:
            for ent in entities:
                if ent["start"] < t["end"] and ent["end"] > t["start"]:
                    hit = ent
                    break
        if hit is None:
            merged.append({"word": normalize_word(t["word"]), "pos": t["pos"]})
            i += 1
            continue
        j = i
        while j < len(abs_tokens):
            tj = abs_tokens[j]
            if tj["start"] < 0 or not (hit["start"] < tj["end"] and hit["end"] > tj["start"]):
                break
            j += 1
        merged.append({"word": None, "type": hit["type"], "ids": hit["ids"], "pos": "NN"})
        i = max(j, i + 1)
    return merged


def prepare_document(doc):
    """Adds doc['sentences'] and doc['sent_of'].

    Title is always sentence 0. Abstract sentences follow (1..). One NLP call per side.
    """
    entities = sorted(doc["entities"], key=lambda e: (e["start"], e["end"]))

    def annotate_side(text, base_offset):
        if not text:
            return []
        out = []
        for sent in NLP.annotate(text):
            abs_tokens = []
            for t in sent:
                if t["start"] >= 0:
                    abs_tokens.append({
                        "word": t["word"], "pos": t["pos"],
                        "start": base_offset + t["start"],
                        "end": base_offset + t["end"],
                    })
                else:
                    abs_tokens.append({
                        "word": t["word"], "pos": t["pos"], "start": -1, "end": -1,
                    })
            out.append(_merge_entities_into_sentence(abs_tokens, entities))
        return out

    title_parts = annotate_side(doc["title"], 0)
    if not title_parts:
        title_tokens = [{"word": normalize_word(w), "pos": "NN"} for w in doc["title"].split()]
        title_parts = [title_tokens or [{"word": "<pad>", "pos": "NN"}]]
    # Paper: title = one sentence
    title_sent = [t for s in title_parts for t in s]

    abs_sents = annotate_side(doc["abstract"], len(doc["title"]) + 1)
    doc["sentences"] = [title_sent] + abs_sents

    doc["sent_of"] = defaultdict(set)
    for si, sent in enumerate(doc["sentences"]):
        for t in sent:
            if t.get("word") is None:
                for eid in t["ids"]:
                    doc["sent_of"][eid].add(si)

print("prepare_document defined")

prepare_document defined


In [34]:
# Annotating every document with CoreNLP/Stanza can take several minutes.
for split_name, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    for i, d in enumerate(docs):
        prepare_document(d)
        if (i + 1) % 100 == 0:
            print(f"  {split_name}: {i + 1}/{len(docs)}")
    print(f"{split_name} prepared ({len(docs)})")

  train: 100/500
  train: 200/500
  train: 300/500
  train: 400/500
  train: 500/500
train prepared (500)
  dev: 100/500
  dev: 200/500
  dev: 300/500
  dev: 400/500
  dev: 500/500
dev prepared (500)
  test: 100/500
  test: 200/500
  test: 300/500
  test: 400/500
  test: 500/500
test prepared (500)


In [35]:
def remove_brackets(words, tags):
    """Delete ( ... ) / [ ... ] groups that do not contain a candidate entity."""
    out_w, out_t, buffer = [], [], None
    for w, t in zip(words, tags):
        if buffer is None and w in ("(", "["):
            buffer = [(w, t)]
        elif buffer is not None:
            buffer.append((w, t))
            if w in (")", "]"):
                if any(x in ("ch_start", "ds_start") for x, _ in buffer):
                    out_w += [x for x, _ in buffer]
                    out_t += [y for _, y in buffer]
                buffer = None
        else:
            out_w.append(w)
            out_t.append(t)
    if buffer:
        out_w += [x for x, _ in buffer]
        out_t += [y for _, y in buffer]
    return out_w, out_t


def render(doc, chem=None, dis=None):
    """List of (words, tags) per sentence. If chem/dis given, mark that pair."""
    result = []
    for sentence in doc["sentences"]:
        words, tags = [], []
        for t in sentence:
            if t["word"] is not None:
                words.append(t["word"]); tags.append(t["pos"])
            elif chem is not None and t["type"] == "Chemical" and chem in t["ids"]:
                words += ["ch_start", "C_" + chem, "ch_end"]; tags += ["MARK", "NN", "MARK"]
            elif dis is not None and t["type"] == "Disease" and dis in t["ids"]:
                words += ["ds_start", "D_" + dis, "ds_end"]; tags += ["MARK", "NN", "MARK"]
            else:
                prefix = "C_" if t["type"] == "Chemical" else "D_"
                words.append(prefix + t["ids"][0]); tags.append("NN")
        if chem is not None or dis is not None:
            words, tags = remove_brackets(words, tags)
        result.append((words, tags))
    return result


example = train_docs[0]
chem = next(i for e in example["entities"] if e["type"] == "Chemical" for i in e["ids"])
dis  = next(i for e in example["entities"] if e["type"] == "Disease" for i in e["ids"])
print("PMID", example["pmid"], "pair", chem, dis)
for words, tags in render(example, chem, dis)[:3]:
    print(" ".join(words))

PMID 227508 pair D009270 D006973
ch_start C_D009270 ch_end reverses the antihypertensive effect of C_D003000 .
in unanesthetized , spontaneously ds_start D_D006973 ds_end rats the decrease in blood pressure and heart rate produced by intravenous C_D003000 , <num> to <num> micrograms / kg , was inhibited or reversed by nalozone , <num> to <num> mg / kg .
the D_D007022 effect of <num> mg / kg C_D008750 was also partially reversed by ch_start C_D009270 ch_end .


In [ ]:
def candidate_pairs(doc, use_k_filter):
    chems = {i for e in doc["entities"] if e["type"] == "Chemical" for i in e["ids"]}
    dises = {i for e in doc["entities"] if e["type"] == "Disease" for i in e["ids"]}
    pairs = []
    for c in sorted(chems):
        for d in sorted(dises):
            if c not in doc["sent_of"] or d not in doc["sent_of"]:
                continue
            if not doc["sent_of"][c] or not doc["sent_of"][d]:
                continue
            span = min(abs(i - j) for i in doc["sent_of"][c] for j in doc["sent_of"][d]) + 1
            if use_k_filter and span > K_WINDOW:
                continue
            pairs.append((c, d, int((c, d) in doc["cids"])))
    return pairs


def make_instances(docs, use_k_filter):
    instances = []
    for doc in docs:
        for c, d, label in candidate_pairs(doc, use_k_filter):
            sents = render(doc, c, d)
            instances.append({
                "pmid": doc["pmid"], "chem": c, "dis": d, "label": label,
                "title": sents[0] if sents else ([], []),
                "abstract": sents[1:] if len(sents) > 1 else [],
            })
    return instances



## §3.1 Training and evaluation settings

The original training and development articles (500 + 500) are shuffled and partitioned into **10 equal subsets** of 100 articles. Nine subsets form the new training set and one subset is the new development set. The original test set is left as it is.

This split is over articles, not over chemical–disease pairs. Table 2's candidate counts (9,626 + 962 = 10,588) are not divisible by 10, while 1,000 articles divide evenly.

The minimum sentence span **K = 4** is used only for the new development set and the test set. Training keeps every chemical–disease pair. A true CID whose mentions lie more than K sentences apart is never shown to the model, so it counts only as a false negative. The paper reports 3 such pairs in the new development set and 24 in the test set.

Performance is precision (P), recall (R), and F = (2PR) / (P + R), implemented with Keras.

`UNION_SEED = 100` and development subset 3 are the partition that reproduces Table 2's CID counts (training 1,863, development 187) together with those 3 development relations. Candidate totals still differ slightly from Table 2 where our sentence boundaries differ from the paper's CoreNLP run. On the test set the unfiltered candidate count (5,405) and the CID count (1,066) match Table 2.

Outputs already stored under Part 6–8 come from the previous pair-level 90/10 split. Re-run from this cell through training before comparing those scores with the paper.


In [1]:
# §3.1 — 10 equal article subsets; K filter only on new dev and test.
# Seed 100 / subset 3 matches Table 2 CID counts (1863 / 187) and the
# 3 development CIDs whose minimum sentence span exceeds K.

def iter_cd_pairs(doc):
    """Every concept-level chemical–disease pair that maps onto sentences."""
    chems = {i for e in doc["entities"] if e["type"] == "Chemical" for i in e["ids"]}
    dises = {i for e in doc["entities"] if e["type"] == "Disease" for i in e["ids"]}
    for c in sorted(chems):
        for d in sorted(dises):
            if not doc["sent_of"].get(c) or not doc["sent_of"].get(d):
                continue
            span = min(abs(i - j) for i in doc["sent_of"][c] for j in doc["sent_of"][d]) + 1
            yield c, d, int((c, d) in doc["cids"]), span


def corpus_stats(docs, apply_k):
    cid = original = kept = beyond = 0
    beyond_pairs = []
    for doc in docs:
        for c, d, label, span in iter_cd_pairs(doc):
            original += 1
            cid += label
            if apply_k and span > K_WINDOW:
                if label:
                    beyond += 1
                    beyond_pairs.append((doc["pmid"], c, d))
            else:
                kept += 1
    return {
        "articles": len(docs),
        "cid_pairs": cid,
        "original_cd": original,
        "cd_after_filter": kept,
        "cid_beyond_k": beyond,
        "beyond_pairs": beyond_pairs,
    }


UNION_SEED = 100
DEV_SUBSET = 3

union_docs = list(train_docs) + list(dev_docs)
rng = random.Random(UNION_SEED)
rng.shuffle(union_docs)
assert len(union_docs) % 10 == 0, len(union_docs)
fold = len(union_docs) // 10
subsets = [union_docs[i * fold:(i + 1) * fold] for i in range(10)]
new_dev_docs = subsets[DEV_SUBSET]
new_train_docs = [d for i, subset in enumerate(subsets) if i != DEV_SUBSET for d in subset]

# Training: no K filter. Dev and test: drop pairs with span > K.
train_instances = make_instances(new_train_docs, use_k_filter=False)
dev_instances = make_instances(new_dev_docs, use_k_filter=True)
test_instances = make_instances(test_docs, use_k_filter=True)

rows = [
    ("Training", corpus_stats(new_train_docs, apply_k=False)),
    ("Development", corpus_stats(new_dev_docs, apply_k=True)),
    ("Test", corpus_stats(test_docs, apply_k=True)),
]
dev_beyond_k = rows[1][1]["beyond_pairs"]
test_beyond_k = rows[2][1]["beyond_pairs"]

# Gold used at evaluation: every CID, including those the K filter removed.
gold_dev = {(d["pmid"], c, s) for d in new_dev_docs for (c, s) in d["cids"]}
gold = {(d["pmid"], c, s) for d in test_docs for (c, s) in d["cids"]}

paper = {
    "Training": (1863, 9626, 9626, None),
    "Development": (187, 962, 926, 3),
    "Test": (1066, 5405, 4713, 24),
}

print(f"Partition: {len(subsets)} subsets x {fold} articles "
      f"(seed {UNION_SEED}, development = subset {DEV_SUBSET})")
print(f"Instances fed to the model: "
      f"train {len(train_instances)} (no K filter), "
      f"dev {len(dev_instances)}, test {len(test_instances)}")
print()
header = f"{'Dataset':<14}{'Articles':>10}{'CID pairs':>12}{'Original CD':>14}{'CD after K':>12}{'CID span>K':>12}"
print("Table 2 — this run")
print(header)
for name, row in rows:
    # Training does not apply K, so span>K relations are examples, not false negatives.
    beyond_s = "—" if name == "Training" else str(row["cid_beyond_k"])
    print(f"{name:<14}{row['articles']:>10}{row['cid_pairs']:>12}"
          f"{row['original_cd']:>14}{row['cd_after_filter']:>12}{beyond_s:>12}")
print()
print("Table 2 — paper")
print(header)
for name, (cid, orig, kept, beyond) in paper.items():
    arts = {"Training": 900, "Development": 100, "Test": 500}[name]
    beyond_s = "—" if beyond is None else str(beyond)
    print(f"{name:<14}{arts:>10}{cid:>12}{orig:>14}{kept:>12}{beyond_s:>12}")
print()
print(f"False negatives from span > K: development {len(dev_beyond_k)} (paper 3), "
      f"test {len(test_beyond_k)} (paper 24)")
print(f"Gold CID pairs (includes those false negatives): "
      f"dev {len(gold_dev)}, test {len(gold)}")


Partition: 10 subsets x 100 articles (seed 100, development = subset 3)
Instances fed to the model: train 9656 (no K filter), dev 932, test 4729

Table 2 — this run
Dataset         Articles   CID pairs   Original CD  CD after K  CID span>K
Training             900        1863          9656        9656           —
Development          100         187          1037         932           3
Test                 500        1066          5405        4729          24

Table 2 — paper
Dataset         Articles   CID pairs   Original CD  CD after K  CID span>K
Training             900        1863          9626        9626           —
Development          100         187           962         926           3
Test                 500        1066          5405        4713          24

False negatives from span > K: development 3 (paper 3), test 24 (paper 24)
Gold CID pairs (includes those false negatives): dev 187, test 1066


## Part 4. Encoding (paper §2.2.5)

Word (100-d) + POS (10-d) embeddings. Prefer PubTator-pretrained vectors from
`embeddings/w2v_word.model` / `w2v_pos.model` (see `scripts/build_pubtator_embeddings.py`);
fall back to CDR-only word2vec if those files are missing.


In [ ]:
# train_instances / dev_instances come from §3.1 (K already applied on dev).
# Vocabulary is fit on the new training set only.

word_counts, pos_set = Counter(), set()
for inst in train_instances:
    for words, tags in [inst["title"]] + inst["abstract"]:
        word_counts.update(words)
        pos_set.update(tags)

word2idx = {"<pad>": 0, "<unk>": 1}
for w, _ in word_counts.most_common():
    word2idx[w] = len(word2idx)

pos2idx = {"<pad>": 0, "<unk>": 1}
for p in sorted(pos_set):
    pos2idx[p] = len(pos2idx)

print(len(train_instances), "train /", len(dev_instances), "dev pairs")
print(len(word2idx), "words,", len(pos2idx), "POS tags")

In [38]:
# ---------------------------------------------------------------------------
# Word / POS embeddings — Zheng et al. §3.2 (PubTator pretraining + CDR)
# ---------------------------------------------------------------------------
# Paper: word2vec on ~22 GB PubTator bioconcepts *offsets* texts + CDR;
#        POS2vec on a random ~1/5 of that corpus (~4.6 GB), 43 PTB tags.
#
# NCBI retired bioconcepts2pubtator_offsets.gz. Current equivalent with text:
#   PubTator3 BioCXML  (https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/BioCXML.0.tar.gz)
#
# Build once (long; ~18 GB download + training):
#   source dl/bin/activate
#   python scripts/build_pubtator_embeddings.py --download
# Optional faster partial run: add  --max-docs 100000
# ---------------------------------------------------------------------------

from gensim.models import Word2Vec

EMB_DIR = REPO_ROOT / "embeddings"
WORD_MODEL_PATH = EMB_DIR / "w2v_word.model"
POS_MODEL_PATH = EMB_DIR / "w2v_pos.model"

def _matrix_from_w2v(model, vocab, dim):
    mat = np.random.normal(0, 0.1, (len(vocab), dim)).astype("float32")
    mat[0] = 0
    hits = 0
    for tok, index in vocab.items():
        if tok in model.wv:
            mat[index] = model.wv[tok]
            hits += 1
    return mat, hits


if WORD_MODEL_PATH.is_file():
    print("Loading PubTator(+CDR) word2vec:", WORD_MODEL_PATH)
    w2v = Word2Vec.load(str(WORD_MODEL_PATH))
    embedding_matrix, w_hits = _matrix_from_w2v(w2v, word2idx, WORD_DIM)
    print(f"  word OOV coverage: {w_hits}/{len(word2idx)} ({100*w_hits/max(1,len(word2idx)):.1f}% in-vocab hits)")
else:
    print("WARNING: missing", WORD_MODEL_PATH)
    print("  Falling back to CDR-only word2vec. Run:")
    print("    python scripts/build_pubtator_embeddings.py --download")
    w2v_word_sents = []
    for doc in train_docs + dev_docs + test_docs:
        for words, tags in render(doc, None, None):
            if words:
                w2v_word_sents.append(words)
    w2v = Word2Vec(
        w2v_word_sents, vector_size=WORD_DIM, window=5, min_count=1, epochs=20, seed=42, workers=4
    )
    embedding_matrix, w_hits = _matrix_from_w2v(w2v, word2idx, WORD_DIM)
    print(f"  CDR-only hits: {w_hits}/{len(word2idx)}")

if POS_MODEL_PATH.is_file():
    print("Loading PubTator POS2vec:", POS_MODEL_PATH)
    pos2v = Word2Vec.load(str(POS_MODEL_PATH))
    pos_embedding_matrix, p_hits = _matrix_from_w2v(pos2v, pos2idx, POS_DIM)
    print(f"  POS OOV coverage: {p_hits}/{len(pos2idx)} ({100*p_hits/max(1,len(pos2idx)):.1f}% in-vocab hits)")
else:
    print("WARNING: missing", POS_MODEL_PATH, "— POS2vec on CDR tag sequences")
    w2v_pos_sents = []
    for doc in train_docs + dev_docs + test_docs:
        for words, tags in render(doc, None, None):
            if tags:
                w2v_pos_sents.append(tags)
    pos2v = Word2Vec(
        w2v_pos_sents, vector_size=POS_DIM, window=5, min_count=1, epochs=20, seed=42, workers=4
    )
    pos_embedding_matrix, p_hits = _matrix_from_w2v(pos2v, pos2idx, POS_DIM)
    print(f"  CDR-only POS hits: {p_hits}/{len(pos2idx)}")

print("word emb", embedding_matrix.shape, "| POS emb", pos_embedding_matrix.shape)


Loading PubTator(+CDR) word2vec: /home/oxored/Documents/DocCIDR/embeddings/w2v_word.model
  word OOV coverage: 8709/11006 (79.1% in-vocab hits)
Loading PubTator POS2vec: /home/oxored/Documents/DocCIDR/embeddings/w2v_pos.model
  POS OOV coverage: 46/49 (93.9% in-vocab hits)
word emb (11006, 100) | POS emb (49, 10)


In [39]:
def encode_sentence(items, vocab):
    ids = [vocab.get(x, 1) for x in items][:MAX_WORDS]
    return [0] * (MAX_WORDS - len(ids)) + ids


def encode(instances):
    n = len(instances)
    doc_words = np.zeros((n, MAX_SENTS, MAX_WORDS), dtype="int32")
    doc_pos = np.zeros((n, MAX_SENTS, MAX_WORDS), dtype="int32")
    title_words = np.zeros((n, MAX_WORDS), dtype="int32")
    title_pos = np.zeros((n, MAX_WORDS), dtype="int32")
    labels = np.zeros(n, dtype="int32")

    for k, inst in enumerate(instances):
        sents = inst["abstract"][:MAX_SENTS]
        first_row = MAX_SENTS - len(sents)
        for s, (words, tags) in enumerate(sents):
            doc_words[k, first_row + s] = encode_sentence(words, word2idx)
            doc_pos[k, first_row + s] = encode_sentence(tags, pos2idx)
        tw, tt = inst["title"]
        title_words[k] = encode_sentence(tw, word2idx)
        title_pos[k] = encode_sentence(tt, pos2idx)
        labels[k] = inst["label"]

    return {
        "doc_words": doc_words, "doc_pos": doc_pos,
        "title_words": title_words, "title_pos": title_pos,
    }, labels


X_train, y_train = encode(train_instances)
X_dev, y_dev = encode(dev_instances)
X_test, y_test = encode(test_instances)
print({k: v.shape for k, v in X_train.items()}, "pos rate", float(y_train.mean()))

{'doc_words': (9623, 30, 120), 'doc_pos': (9623, 30, 120), 'title_words': (9623, 120), 'title_pos': (9623, 120)} pos rate 0.1914163982126156


## Part 5. Model (paper §2.2.3–2.2.8, Fig. 1)

Word∥POS → BiLSTM per sentence → BiLSTM over sentences (Dl′) + CNN+max-pool (Dc′) + title BiLSTM (Dt′) → concat → softmax.

In [40]:
# M1 — inputs (clear previous TF graph / free GPU from earlier runs)
keras.backend.clear_session()
gc.collect()

doc_words_in   = keras.Input(shape=(MAX_SENTS, MAX_WORDS), dtype="int32", name="doc_words")
doc_pos_in     = keras.Input(shape=(MAX_SENTS, MAX_WORDS), dtype="int32", name="doc_pos")
title_words_in = keras.Input(shape=(MAX_WORDS,), dtype="int32", name="title_words")
title_pos_in   = keras.Input(shape=(MAX_WORDS,), dtype="int32", name="title_pos")
print("session cleared; inputs ready")


session cleared; inputs ready


In [41]:
# M2 — embedding (word 100 + POS 10 = 110)
word_embedding = layers.Embedding(
    len(word2idx), WORD_DIM,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=True, name="word_embedding",
)
pos_embedding = layers.Embedding(
    len(pos2idx), POS_DIM,
    embeddings_initializer=keras.initializers.Constant(pos_embedding_matrix),
    trainable=True, name="pos_embedding",
)

doc_x = layers.Concatenate(name="doc_word_pos")([
    word_embedding(doc_words_in), pos_embedding(doc_pos_in)
])
title_x = layers.Concatenate(name="title_word_pos")([
    word_embedding(title_words_in), pos_embedding(title_pos_in)
])

In [42]:
# M3 — BiLSTM1 over words of each sentence -> S'_i (220)
bilstm_words = layers.Bidirectional(layers.LSTM(110), name="bilstm1_words")
sentence_vectors = layers.TimeDistributed(bilstm_words, name="each_sentence")(doc_x)
sentence_vectors = layers.Dropout(DROPOUT)(sentence_vectors)

In [43]:
# M4 — BiLSTM2 over sentence sequence -> Dl' (440)
D_l = layers.Bidirectional(layers.LSTM(220), name="bilstm2_sentences")(sentence_vectors)
D_l = layers.Dropout(DROPOUT)(D_l)

In [44]:
# M5 — CNN window=5 over sentences + max-pool -> Dc' (300)
conv = layers.Conv1D(
    CNN_FILTERS, CNN_WINDOW, activation="relu", padding="same", name="cnn_sentences"
)(sentence_vectors)
D_c = layers.GlobalMaxPooling1D(name="max_pool")(conv)

In [45]:
# M6 — BiLSTM3 on title -> Dt' (220)
D_t = layers.Bidirectional(layers.LSTM(110), name="bilstm3_title")(title_x)
D_t = layers.Dropout(DROPOUT)(D_t)

In [46]:
# M7 — concatenate three views + softmax
merged = layers.Concatenate(name="Dl_Dc_Dt")([D_l, D_c, D_t])
merged = layers.Dropout(DROPOUT)(merged)
output = layers.Dense(2, activation="softmax", name="softmax")(merged)

In [47]:
# M8 — compile (RMSprop lr=0.001, rho=0.9; cross-entropy)
model = keras.Model(
    inputs=[doc_words_in, doc_pos_in, title_words_in, title_pos_in],
    outputs=output, name="cid_lstm_cnn",
)
model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=LEARNING_RATE, rho=0.9),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


Model: "cid_lstm_cnn"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ doc_words           │ (None, 30, 120)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ doc_pos             │ (None, 30, 120)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 120, 100)  │  1,100,600 │ doc_words[0][0],  │
│ (Embedding)         │                   │            │ title_words[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pos_embedding       │ (None, 120, 10)   │        490 │ doc_pos[0][0],    │
│ (Embedding)         │                   │            │ title_pos[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ doc_word_pos        │ (None, 30, 120,   │          0 │ word_embedding[0… │
│ (Concatenate)       │ 110)              │            │ pos_embedding[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ title_words         │ (None, 120)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ title_pos           │ (None, 120)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ each_sentence       │ (None, 30, 220)   │    194,480 │ doc_word_pos[0][… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 30, 220)   │          0 │ each_sentence[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ title_word_pos      │ (None, 120, 110)  │          0 │ word_embedding[1… │
│ (Concatenate)       │                   │            │ pos_embedding[1]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm2_sentences   │ (None, 440)       │    776,160 │ dropout[0][0]     │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_sentences       │ (None, 30, 300)   │    330,300 │ dropout[0][0]     │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm3_title       │ (None, 220)       │    194,480 │ title_word_pos[0… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 440)       │          0 │ bilstm2_sentence… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pool            │ (None, 300)       │          0 │ cnn_sentences[0]… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 220)       │          0 │ bilstm3_title[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dl_Dc_Dt            │ (None, 960)       │          0 │ dropout_1[0][0],  │
│ (Concatenate)       │                   │            │ max_pool[0][0],   │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 2,598,432 (9.91 MB)

 Trainable params: 2,598,432 (9.91 MB)

 Non-trainable params: 0 (0.00 B)

## Part 6. Training

Cross-entropy, RMSprop (lr 0.001, rho 0.9), dropout 0.5 (paper).

Recall-oriented changes vs a plain paper clone:
- **class weights** (up-weight CID positives)
- **longer training** (`MAX_EPOCHS=40`, patience 8) + ReduceLROnPlateau
- **batch 4** + `clear_session` before rebuild to free GPU memory
- after training, **threshold sweep on dev** (paper used 0.5)


**Retrain result** (`scripts/train_cid_recall.py`, early-stop epoch 1/9, thr=0.3):
- Dev: P/R/F = 39.0 / 60.7 / 47.5
- Test no-PP: **33.2 / 57.9 / 42.2** (was ~55.8 / 30.0 / 39.0)
- Test +rule1: 33.8 / 61.9 / 43.7
- Paper Table5 no-PP: 54.3 / 65.9 / 59.5 | Table7+pp: 56.2 / 68.0 / 61.5
- Model: `embeddings/cid_model_recall/`


In [48]:
# Class weights: CID positives are ~19% → without weights the model becomes too precise / low-recall
n_neg, n_pos = np.bincount(y_train, minlength=2)
if POS_WEIGHT is None:
    # balanced heuristic
    w0 = (n_neg + n_pos) / (2.0 * max(n_neg, 1))
    w1 = (n_neg + n_pos) / (2.0 * max(n_pos, 1))
else:
    w0, w1 = 1.0, float(POS_WEIGHT)
class_weight = {0: float(w0), 1: float(w1)}
print(f"train counts neg={n_neg} pos={n_pos} | class_weight={class_weight}")

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=max(2, PATIENCE // 3),
        min_lr=1e-5, verbose=1,
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_dev, y_dev),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)
print("epochs run:", len(history.history["loss"]))


train counts neg=7781 pos=1842 | class_weight={0: 0.6183652486826886, 1: 2.6121064060803474}
Epoch 1/40
2406/2406 ━━━━━━━━━━━━━━━━━━━━ 489s 196ms/step - accuracy: 0.6049 - loss: 0.7741 - val_accuracy: 0.7505 - val_loss: 0.5086 - learning_rate: 0.0010
Epoch 2/40
2406/2406 ━━━━━━━━━━━━━━━━━━━━ 467s 194ms/step - accuracy: 0.6501 - loss: 0.9331 - val_accuracy: 0.7613 - val_loss: 0.5325 - learning_rate: 0.0010
Epoch 3/40
2406/2406 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step - accuracy: 0.6854 - loss: 0.7657
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
2406/2406 ━━━━━━━━━━━━━━━━━━━━ 461s 192ms/step - accuracy: 0.6854 - loss: 0.7657 - val_accuracy: 0.7796 - val_loss: 0.5346 - learning_rate: 0.0010
Epoch 4/40
2406/2406 ━━━━━━━━━━━━━━━━━━━━ 461s 191ms/step - accuracy: 0.7396 - loss: 0.6767 - val_accuracy: 0.7688 - val_loss: 0.5573 - learning_rate: 5.0000e-04
Epoch 5/40
2406/2406 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step - accuracy: 0.7565 - loss: 0.6552
Epoch 5: ReduceLROnPlateau

## Part 7. Evaluation

Precision, recall, and F = (2PR) / (P + R), as in §3.1. Gold is every document-level CID, so a relation whose minimum sentence span exceeds K is a false negative.

The decision threshold is then swept on the new development set (the paper fixed p = 0.5) and applied to the test set.


In [49]:
def predict_pairs(instances, probs, thr):
    out = set()
    for inst, p in zip(instances, probs):
        if p >= thr:
            out.add((inst["pmid"], inst["chem"], inst["dis"]))
    return out


def prf(pred, gold_set):
    tp = len(pred & gold_set)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(gold_set) if gold_set else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return round(100 * p, 1), round(100 * r, 1), round(100 * f, 1)


# Probabilities for CID class (=1)
dev_probs = model.predict(X_dev, batch_size=32)[:, 1]
test_probs = model.predict(X_test, batch_size=32)[:, 1]

# §3.1 gold: every CID in the article set. Pairs with span > K are not
# candidates, so they contribute only as false negatives.
gold_dev = {(d["pmid"], c, s) for d in new_dev_docs for (c, s) in d["cids"]}
gold = {(d["pmid"], c, s) for d in test_docs for (c, s) in d["cids"]}
print(
    f"Dev gold {len(gold_dev)} CIDs, {len(dev_beyond_k)} with span > K (automatic FN); "
    f"test gold {len(gold)} CIDs, {len(test_beyond_k)} with span > K"
)

print("Dev threshold sweep (maximize F1; break ties toward higher recall):")
best = None
for thr in THRESH_GRID:
    pred = predict_pairs(dev_instances, dev_probs, thr)
    scores = prf(pred, gold_dev)
    print(f"  thr={thr:.2f}  P/R/F={scores}  n={len(pred)}")
    # key: F1, then recall, then lower thr
    key = (scores[2], scores[1], -thr)
    if best is None or key > best[0]:
        best = (key, thr, scores)

THRESHOLD = best[1]
print(f"\nSelected THRESHOLD={THRESHOLD:.2f} from dev (P/R/F={best[2]})")

predicted = predict_pairs(test_instances, test_probs, THRESHOLD)
print(len(predicted), "test pairs predicted as CID @", THRESHOLD)


30/30 ━━━━━━━━━━━━━━━━━━━━ 9s 223ms/step
148/148 ━━━━━━━━━━━━━━━━━━━━ 16s 108ms/step
Dev threshold sweep (maximize F1; break ties toward higher recall):
  thr=0.50  P/R/F=(39.2, 27.9, 32.6)  n=143
  thr=0.45  P/R/F=(38.5, 33.3, 35.7)  n=174
  thr=0.40  P/R/F=(38.1, 38.3, 38.2)  n=202
  thr=0.35  P/R/F=(37.3, 44.8, 40.7)  n=241
  thr=0.30  P/R/F=(37.4, 53.2, 43.9)  n=286
  thr=0.25  P/R/F=(35.9, 59.7, 44.9)  n=334

Selected THRESHOLD=0.25 from dev (P/R/F=(35.9, 59.7, 44.9))
1969 test pairs predicted as CID @ 0.25


In [50]:
print("Without post-processing  P / R / F:", prf(predicted, gold))
print("Paper (Table 5, row 1):            (54.3, 65.9, 59.5)")
print("Also @ paper thr=0.5:", prf(predict_pairs(test_instances, test_probs, 0.5), gold))


Without post-processing  P / R / F: (31.7, 58.6, 41.2)
Paper (Table 5, row 1):            (54.3, 65.9, 59.5)
Also @ paper thr=0.5: (39.5, 25.8, 31.2)


In [51]:
doc_by_pmid = {d["pmid"]: d for d in test_docs}

def is_intra(pmid, c, d):
    doc = doc_by_pmid[pmid]
    return len(doc["sent_of"].get(c, set()) & doc["sent_of"].get(d, set())) > 0

for name, wanted in [("Intra-sentential", True), ("Inter-sentential", False)]:
    g = {x for x in gold if is_intra(*x) == wanted}
    p = {x for x in predicted if is_intra(*x) == wanted}
    print(name, prf(p, g))

Intra-sentential (39.0, 64.9, 48.7)
Inter-sentential (19.2, 43.8, 26.7)


## Part 8. Post-processing (paper §2.2.9)

- **Rule 1:** if an article has no predicted CID, emit the most frequent chemical × most frequent disease.
- **Rule 2:** drop diseases that are more general than another disease in the same article (MeSH tree), unless the disease appears in the title.

In [52]:
def rule_1(pred, docs):
    new = set(pred)
    found = {pmid for pmid, _, _ in pred}
    for doc in docs:
        if doc["pmid"] in found:
            continue
        chem_count = Counter(
            eid for e in doc["entities"] if e["type"] == "Chemical" for eid in e["ids"]
        )
        dis_count = Counter(
            eid for e in doc["entities"] if e["type"] == "Disease" for eid in e["ids"]
        )
        if chem_count and dis_count:
            new.add((doc["pmid"], chem_count.most_common(1)[0][0], dis_count.most_common(1)[0][0]))
    return new

pred_rule1 = rule_1(predicted, test_docs)
print("+ rule 1  P / R / F:", prf(pred_rule1, gold))

+ rule 1  P / R / F: (32.6, 63.2, 43.0)


In [53]:
import xml.etree.ElementTree as ET

MESH_CANDIDATES = [
    "https://nlmpubs.nlm.nih.gov/projects/mesh/MESH_FILES/xmlmesh/desc2026.xml",
    "https://nlmpubs.nlm.nih.gov/projects/mesh/MESH_FILES/xmlmesh/desc2025.xml",
    "https://nlmpubs.nlm.nih.gov/projects/mesh/MESH_FILES/xmlmesh/desc2024.xml",
]
MESH_PATH = REPO_ROOT / "desc.xml"

if not MESH_PATH.exists():
    last_err = None
    for url in MESH_CANDIDATES:
        try:
            print("Downloading", url)
            urllib.request.urlretrieve(url, MESH_PATH)
            break
        except Exception as e:
            last_err = e
            print(" failed:", e)
    else:
        raise RuntimeError(f"Could not download MeSH XML: {last_err}")

tree_numbers = {}
for _, elem in ET.iterparse(MESH_PATH):
    if elem.tag == "DescriptorRecord":
        mesh_id = elem.findtext("DescriptorUI")
        tree_numbers[mesh_id] = [t.text for t in elem.iter("TreeNumber") if t.text]
        elem.clear()
print(len(tree_numbers), "MeSH descriptors")

31110 MeSH descriptors


In [54]:
def more_general(a, b):
    return any(
        tb.startswith(ta + ".")
        for ta in tree_numbers.get(a, [])
        for tb in tree_numbers.get(b, [])
    )


def rule_2(pred, docs):
    new = set(pred)
    for doc in docs:
        diseases = {i for e in doc["entities"] if e["type"] == "Disease" for i in e["ids"]}
        in_title = {
            i for e in doc["entities"]
            if e["type"] == "Disease" and e["start"] < len(doc["title"])
            for i in e["ids"]
        }
        for d in diseases - in_title:
            if any(more_general(d, other) for other in diseases if other != d):
                new = {x for x in new if not (x[0] == doc["pmid"] and x[2] == d)}
    return new

pred_final = rule_2(pred_rule1, test_docs)
print("Final (both rules)  P / R / F:", prf(pred_final, gold))
print("Paper (Table 7):               (56.2, 68.0, 61.5)")

Final (both rules)  P / R / F: (34.8, 61.3, 44.3)
Paper (Table 7):               (56.2, 68.0, 61.5)


## Notes vs the paper

| Point | Paper | This notebook |
| --- | --- | --- |
| Sentence / POS | Stanford CoreNLP | CoreNLP if `CORENLP_HOME`+Java; else Stanza |
| Word vectors | word2vec on ~22 GB PubTator offsets + CDR | `scripts/build_pubtator_embeddings.py` on PubTator3 **BioCXML** + CDR (offsets file retired by NCBI) |
| POS vectors | word2vec on ~1/5 tagged PubTator text | same script: Stanza POS on a 1/5 sample (capped), else CDR fallback |
| Train/dev split | 10 equal article subsets of train∪dev; K only on new dev and test | same (§3.1); seed 100, subset 3 matches Table 2 CID counts |
| Relations with span > K | false negatives (3 dev, 24 test) | counted in gold; test matches 24, dev matches 3 |
| Epochs | not stated | early stopping on dev loss |
| Hidden sizes | 220 / 440 / 220 outputs | BiLSTM(110)/ (220)/ (110) bidirectional |

Build embeddings (once):

```bash
source dl/bin/activate
python scripts/build_pubtator_embeddings.py --download
```

Exploration of the raw CDR files remains in `Proj2.ipynb`.
